In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator # type: ignore
from tensorflow.keras import layers, models # type: ignore
from tensorflow.keras.models import Model, load_model # type: ignore
import matplotlib.pyplot as plt
import os
import json
from PIL import Image

In [ ]:
print("--- GPU Detection Status ---")
print(tf.config.list_physical_devices('GPU'))
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

In [ ]:
source_dirs = 'health_classifier'

In [ ]:
#Step 3: Data generators with augmentations
image_size = 128
batch_size = 32
train_dir = 'health_classifier/train'
val_dir = 'health_classifier/val'

# --- 1. Define Augmentation without validation_split ---
train_datagen = ImageDataGenerator(
    rescale=1./255,
    brightness_range=[0.8, 1.2],
    channel_shift_range=0.2,
    zoom_range=0.3,
    rotation_range=30,
    horizontal_flip=True,
    vertical_flip=True,
    width_shift_range=0.2,
    height_shift_range=0.2,
    fill_mode='nearest'
)

# --- 2. Define Validation Generator (Only Rescaling) ---
val_datagen = ImageDataGenerator(rescale=1./255) 

# --- 3. Create Training Generator ---
train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=(image_size, image_size),
    batch_size=batch_size,
    class_mode='categorical',
)

# --- 4. Create Validation Generator ---
val_gen = val_datagen.flow_from_directory(
    val_dir,
    target_size=(image_size, image_size),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

num_classes = train_gen.num_classes
print(f"Number of classes detected: {num_classes}")
print(f"Classes found by generator: {train_gen.class_indices}")

In [ ]:
# Count samples per class
total_diseased = len(os.listdir('health_classifier/train/diseased'))
total_healthy = len(os.listdir('health_classifier/train/healthy'))
total_samples = total_diseased + total_healthy

print(f"Diseased samples: {total_diseased}")
print(f"Healthy samples: {total_healthy}")
print(f"Total samples: {total_samples}")

# Calculate weights (give more weight to underrepresented class)
weight_diseased = total_samples / (2 * total_diseased)
weight_healthy = total_samples / (2 * total_healthy)

class_weights = {
    0: weight_diseased,    # diseased (usually smaller)
    1: weight_healthy      # healthy
}

print(f"\nClass weights:")
print(f"  Diseased: {weight_diseased:.4f}")
print(f"  Healthy: {weight_healthy:.4f}")

In [ ]:
import keras_tuner as kt
from tensorflow.keras.optimizers import Adam # type: ignore
from tensorflow.keras.layers import BatchNormalization # type: ignore #
from tensorflow.keras import layers, models # type: ignore
import json

# Define the model architecture for hyperparameter tuning
def build_model(hp):
    model = models.Sequential()
    
    # --- Conv Block 1 ---
    model.add(layers.Conv2D(
        filters=hp.Int('conv_1_filters', 32, 128, step=32),
        kernel_size=(3, 3),
        activation='relu',
        input_shape=(image_size, image_size, 3)
    ))
    model.add(BatchNormalization()) 
    model.add(layers.MaxPooling2D(2, 2))
    
    # --- Conv Block 2 ---
    model.add(layers.Conv2D(
        filters=hp.Int('conv_2_filters', 64, 256, step=64),
        kernel_size=(3, 3),
        activation='relu'
    ))
    model.add(BatchNormalization())
    model.add(layers.MaxPooling2D(2, 2))
    
    # --- Conv Block 3 ---
    model.add(layers.Conv2D(
        filters=hp.Int('conv_3_filters', 128, 512, step=128),
        kernel_size=(3, 3),
        activation='relu'
    ))
    model.add(BatchNormalization())
    model.add(layers.MaxPooling2D(2, 2))
    
    # --- Conv Block 4 ---
    model.add(layers.Conv2D(
        filters=hp.Int('conv_4_filters', 128, 512, step=128),
        kernel_size=(3, 3),
        activation='relu',
        padding='same'
    ))
    model.add(BatchNormalization())
    model.add(layers.MaxPooling2D(2, 2))
    
    model.add(layers.Flatten())
    
    # --- Dense Block 1 ---
    model.add(layers.Dense(
        hp.Int('dense_units', 128, 512, step=128),
        activation='relu'
    ))
    model.add(layers.Dropout(hp.Float('dropout', 0.2, 0.5, step=0.1)))
    
    # --- Dense Block 2 ---
    model.add(layers.Dense(
        hp.Int('dense_units_2', 64, 256, step=64),
        activation='relu'
    ))
    model.add(layers.Dropout(0.3))
    
    model.add(layers.Dense(num_classes, activation='softmax'))
    
    model.compile(
        optimizer=Adam(learning_rate=hp.Float('learning_rate', 1e-5, 1e-3, sampling='log')),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Initialize the BayesianOptimization tuner
tuner = kt.BayesianOptimization(
    build_model,
    objective='val_accuracy',
    max_trials=10,
    executions_per_trial=1,
    directory='keras_tuner_dir',
    project_name='HelpTheGreen_v2',
    overwrite=True
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=3,
    restore_best_weights=True
)

print("Starting Hyperparameter Search (10 Trials x 10 Epochs each)...")
tuner.search(train_gen, epochs=10, validation_data=val_gen, class_weight=class_weights,
                callbacks=[early_stop]
)
print("Hyperparameter Search Complete.")

# Get the best model and train final model
best_hps = tuner.get_best_hyperparameters(1)[0]
print("\nBest hyperparameters found:")
for param in best_hps.values:
    print(f"{param}: {best_hps.get(param)}")
model = tuner.hypermodel.build(best_hps)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7
    ),
    tf.keras.callbacks.ModelCheckpoint(
        'plantvillage_tuned_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]
print("\nStarting Final Model Training (100 Epochs)...")
history = model.fit(train_gen, epochs=100, validation_data=val_gen, class_weight=class_weights, callbacks=callbacks)

# Save class labels
class_labels = {v: k for k, v in train_gen.class_indices.items()}
with open('class_labels_combined.json', 'w') as f:
    json.dump(class_labels, f)

print("Model and class labels saved.")

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

def plot_history(history):
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs_range = range(len(acc))

    plt.figure(figsize=(14, 6))

    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label='Training Accuracy')
    plt.plot(epochs_range, val_acc, label='Validation Accuracy')
    plt.legend()
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')

    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label='Training Loss')
    plt.plot(epochs_range, val_loss, label='Validation Loss')
    plt.legend()
    plt.title('Training and Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')

    plt.tight_layout()
    plt.show()

if 'history' in dir():
    plot_history(history)
else:
    print("No history found. Run the training cell first.")

In [ ]:
for epoch in range(len(history.history['accuracy'])):
    print(f"Epoch {epoch+1}:")
    print(f"  Training Accuracy:  {history.history['accuracy'][epoch]:.4f}")
    print(f"  Validation Accuracy:{history.history['val_accuracy'][epoch]:.4f}")
    print(f"  Training Loss:      {history.history['loss'][epoch]:.4f}")
    print(f"  Validation Loss:    {history.history['val_loss'][epoch]:.4f}")


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report # type: ignore
import seaborn as sns # type: ignore

# 1. Get True Labels and Predictions from the Validation Set
print("Generating predictions on validation set...")

# Reset generator to ensure it starts from the beginning
val_gen.reset() 

# Predict the categories for all validation images
# steps = total number of validation samples / batch size
val_steps = val_gen.samples // val_gen.batch_size 
if val_gen.samples % val_gen.batch_size != 0:
    val_steps += 1
    
Y_pred = model.predict(val_gen, steps=val_steps)
y_pred_classes = np.argmax(Y_pred, axis=1) # Predicted classes (e.g., 0, 1)

# Get true classes (ensure generator is not shuffled)
y_true = val_gen.classes # True classes (e.g., 0, 1)

# Get class labels from the generator
class_names = list(train_gen.class_indices.keys())

# 2. Compute Confusion Matrix
conf_matrix = confusion_matrix(y_true, y_pred_classes)

# 3. Print Classification Report (Precision, Recall, F1-score)
print("\n--- Classification Report ---")
print(classification_report(y_true, y_pred_classes, target_names=class_names))

# 4. Plot Confusion Matrix (for report visualization)
plt.figure(figsize=(8, 6))
sns.heatmap(
    conf_matrix, 
    annot=True, 
    fmt='d', 
    cmap='Blues', 
    xticklabels=class_names, 
    yticklabels=class_names
)
plt.title('Confusion Matrix for CNN Model')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

In [ ]:
# ===== FIND OPTIMAL THRESHOLD =====
from sklearn.metrics import precision_recall_curve, roc_curve, auc
import matplotlib.pyplot as plt

# Get probability predictions for healthy class (index 1)
val_gen.reset()
val_steps = -(-val_gen.samples // val_gen.batch_size)  # ceiling division
y_proba = model.predict(val_gen, steps=val_steps)[:, 1]
y_proba = y_proba[:len(y_true)]

# Calculate precision-recall curve
precisions, recalls, thresholds = precision_recall_curve(y_true, y_proba)

# Plot Precision-Recall Curve
plt.figure(figsize=(10, 6))
plt.plot(thresholds, precisions[:-1], 'b-', label='Precision')
plt.plot(thresholds, recalls[:-1], 'r-', label='Recall')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Precision-Recall vs Threshold')
plt.legend()
plt.grid(True)
plt.show()

# Find optimal threshold (where precision and recall are balanced)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
optimal_idx = np.argmax(f1_scores[:-1])
optimal_threshold = thresholds[optimal_idx]

print(f"\n✅ Optimal Threshold: {optimal_threshold:.4f}")
print(f"   - Precision: {precisions[optimal_idx]:.2%}")
print(f"   - Recall: {recalls[optimal_idx]:.2%}")
print(f"   - F1-Score: {f1_scores[optimal_idx]:.2%}")

# Calculate ROC-AUC
fpr, tpr, _ = roc_curve(y_true, y_proba)
roc_auc = auc(fpr, tpr)

# Plot ROC Curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

print(f"\n📊 ROC-AUC Score: {roc_auc:.4f}")

import json
threshold_data = {"optimal_threshold": float(optimal_threshold)}
with open('optimal_threshold.json', 'w') as f:
    json.dump(threshold_data, f)
print(f"Threshold saved: {optimal_threshold:.4f}")

In [ ]:
import numpy as np
from PIL import Image
from tensorflow.keras.models import load_model # type: ignore
model = load_model('plantvillage_tuned_model.h5')
def quick_predict(image_path):
     img = Image.open(image_path).convert("RGB").resize((128, 128))
     arr = np.expand_dims(np.array(img, dtype='float32') / 255.0, axis=0)
     pred = model.predict(arr, verbose=0)[0]
     if pred[1] < optimal_threshold:
          return "🔴 DISEASED"
     return "🟢 HEALTHY"
# Test on your image
TEST_IMAGE = r"C:\Users\majum\Downloads\images (9).jpeg"  # change this
print(quick_predict(TEST_IMAGE))